In [1]:
!pip install konlpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 57.7 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
!apt update

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy InRelease [270 kB]                
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]      
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1581 B]
Get:6 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [61.6 kB]33m
Get:7 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3915 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7004 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1294 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy/restricted amd64 Packages [164 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy/universe amd64 Packages [17.5 MB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jam

In [3]:
!apt install -y openjdk-17-jdk

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  adwaita-icon-theme alsa-topology-conf alsa-ucm-conf at-spi2-core
  ca-certificates-java dbus-user-session dconf-gsettings-backend dconf-service
  fontconfig fonts-dejavu-extra gsettings-desktop-schemas
  gtk-update-icon-cache hicolor-icon-theme humanity-icon-theme java-common
  libasound2 libasound2-data libatk-bridge2.0-0 libatk-wrapper-java
  libatk-wrapper-java-jni libatk1.0-0 libatk1.0-data libatspi2.0-0
  libavahi-client3 libavahi-common-data libavahi-common3 libcairo-gobject2
  libcairo2 libcups2 libdatrie1 libdconf1 libfontenc1 libfribidi0
  libgail-common libgail18 libgdk-pixbuf-2.0-0 libgdk-pixbuf2.0-bin
  libgdk-pixbuf2.0-common libgif7 libgraphite2-3 libgtk2.0-0 libgtk2.0-bin
  libgtk2.0-common libharfbuzz0b libice-dev libice6 liblcms2-2 libnspr4
  libnss3 libpango-1.0-0 libpangocairo-1.0-0 libpangoft2-1.0-0 libpcsclite1
  li

In [5]:
!pip install pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 50.4 MB/s eta 0:00:00:00:01

[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [6]:
from konlpy.tag import Okt
import pandas as pd
okt = Okt()

In [ ]:
import re
df = pd.read_csv('./daum_movie_review.csv')
df['target'] = df['rating'].apply(lambda x : 1 if x>0.5 else 0)
df['clean'] = df['review'].apply(lambda x : re.sub(r'[^가-힣\s]','',x) )    

okt = Okt()
def kor_tokenizer(text):    
    return [
        word for word, pos in okt.pos(text,stem=True) 
            if pos in ['Noun','Verb','Adjective'] and len(word) >=2        
           ]

# 단어사전 구축
from collections import Counter
vocab = Counter()
for text in df['clean']:
    vocab.update(kor_tokenizer(text))

# 10000개의 단어로만 구성
vocab_size = 10000
word_to_index = {
    word:idx+2 for idx, (word,seq) in enumerate(vocab.most_common(vocab_size))
}
word_to_index['<PAD>'] = 0
word_to_index['<UNK>'] = 1

def word2Sequence(text):
    return [word_to_index.get(word,1) for word in kor_tokenizer(text)]    

index_to_word = {idx : word for word, idx in word_to_index.items()}
# padding
import torch
from torch.nn.utils.rnn import pad_sequence
x1 = word2Sequence(df['clean'][0])
x2 = word2Sequence(df['clean'][1])
x1_tensor = torch.LongTensor(x1)
x2_tensor = torch.LongTensor(x2)
print(x1_tensor,x2_tensor)
pad_sequence([x1_tensor,x2_tensor], batch_first=True, padding_value=0)

MAX_LEN = 500
def padding(texts):
    result = []
    for text in texts:
        text = word2Sequence(text)[:MAX_LEN]
        text = torch.LongTensor(text)
        result.append(text)
    return pad_sequence(result,batch_first=True, padding_value=0)
    
padding(df['clean'])  


In [11]:
import pandas as pd
# https://github.com/skn29teacher/skn29_lecture/blob/main/data_nlp/daum_movie_review.csv
url = 'https://raw.githubusercontent.com/skn29teacher/skn29_lecture/main/data_nlp/daum_movie_review.csv'
pd.read_csv(url)

,review,rating,date,title
0,돈 들인건 티가 나지만 보는 내내 하품만,1,2018.10.29,인피니티 워
1,몰입할수밖에 없다. 어렵게 생각할 필요없다. 내가 전투에 참여한듯 손에 땀이남.,10,2018.10.26,인피니티 워
2,이전 작품에 비해 더 화려하고 스케일도 커졌지만.... 전국 맛집의 음식들을 한데 ...,8,2018.10.24,인피니티 워
3,이 정도면 볼만하다고 할 수 있음!,8,2018.10.22,인피니티 워
4,재미있다,10,2018.10.20,인피니티 워
...,...,...,...,...
14720,어른들을 위한 동화 정말 오랜만에 좋은 애니를 보았습니다 가족의 소중...,10,2018.01.12,코코
14721,디즈니는 못해도 본전은 한다.,7,2018.01.12,코코
14722,가족을 위한 영화... 괜찮은 영화.~~~,8,2018.01.12,코코
14723,간만에 제대로 잘짜여진 각본의 영화를 봤네 여운이 아직도 남아~어른을 위한 애니~,10,2018.01.12,코코
